# Deterministic BF-BOF + post-combustion CCS steel NPV

Calculate a single BF-BOF + post-combustion CCS retrofit NPV case using the shared deterministic steel source code. The notebook resolves representative retrofit changes against the parent BAU technology and displays the resolved inputs, BAU-relative changes, and processed financial outputs.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from steel.steel_npv_deterministic import calculate_deterministic_steel_result

pd.options.display.float_format = "{:,.3f}".format

In [2]:
TECHNOLOGY = "bf_bof_post_combustion_ccs"

result = calculate_deterministic_steel_result(TECHNOLOGY)
value = {key: values[0] for key, values in result.items()}

UNITS = {
    "annual_output_tcs": "tCS/year",
    "lifetime_years": "years",
    "technology_type": "category",
    "capex_eur_per_tcs": "EUR/tCS",
    "fixed_opex_eur_per_tcs": "EUR/tCS",
    "variable_opex_eur_per_tcs": "EUR/tCS",
    "fuel_type": "category",
    "fuel_consumption_mwh_th_per_tcs": "MWh_th/tCS",
    "hydrogen_consumption_kg_per_tcs": "kg/tCS",
    "charcoal_consumption_mwh_th_per_tcs": "MWh_th/tCS",
    "fuel_price_eur_per_mwh_th": "EUR/MWh_th",
    "green_hydrogen_price_eur_per_kg": "EUR/kg",
    "charcoal_price_eur_per_mwh_th": "EUR/MWh_th",
    "electricity_consumption_mwh_per_tcs": "MWh/tCS",
    "electricity_price_eur_per_mwh": "EUR/MWh",
    "emissions_tco2_per_tcs": "tCO2/tCS",
    "steel_price_eur_per_tcs": "EUR/tCS",
    "carbon_price_eur_per_t": "EUR/tCO2",
    "transport_and_storage_share_of_capture_cost": "fraction",
    "capex_change_eur_per_tcs": "EUR/tCS",
    "fixed_opex_change_eur_per_tcs": "EUR/tCS",
    "variable_opex_change_eur_per_tcs": "EUR/tCS",
    "fuel_consumption_reduction_fraction": "fraction",
    "electricity_consumption_reduction_fraction": "fraction",
    "emissions_reduction_fraction": "fraction",
    "initial_capex_eur": "EUR",
    "annual_revenue_eur": "EUR/year",
    "annual_fixed_opex_eur": "EUR/year",
    "annual_variable_opex_eur": "EUR/year",
    "annual_pci_coking_coal_cost_eur": "EUR/year",
    "annual_charcoal_cost_eur": "EUR/year",
    "annual_natural_gas_cost_eur": "EUR/year",
    "annual_hydrogen_cost_eur": "EUR/year",
    "annual_fuel_cost_eur": "EUR/year",
    "annual_electricity_cost_eur": "EUR/year",
    "capture_cost_excluding_transport_and_storage_eur_per_tcs": "EUR/tCS",
    "transport_and_storage_cost_eur_per_tcs": "EUR/tCS",
    "annual_transport_and_storage_cost_eur": "EUR/year",
    "annual_emissions_cost_eur": "EUR/year",
    "annual_total_cost_eur": "EUR/year",
    "annual_net_cash_flow_eur": "EUR/year",
    "npv_eur": "EUR",
    "discounted_lifetime_output_tcs": "tCS",
    "present_value_total_cost_eur": "EUR",
    "lcos_eur_per_tcs": "EUR/tCS",
    "levelized_net_margin_eur_per_tcs": "EUR/tCS",
}

ENERGY_INPUT_COLUMNS = {
    "h2_dri_eaf": (
        "hydrogen_consumption_kg_per_tcs",
        "green_hydrogen_price_eur_per_kg",
        "charcoal_consumption_mwh_th_per_tcs",
        "charcoal_price_eur_per_mwh_th",
    ),
}.get(
    TECHNOLOGY,
    (
        "fuel_consumption_mwh_th_per_tcs",
        "fuel_price_eur_per_mwh_th",
    ),
)

RAW_INPUT_COLUMNS = (
    "annual_output_tcs",
    "lifetime_years",
    "technology_type",
    "capex_eur_per_tcs",
    "fixed_opex_eur_per_tcs",
    "variable_opex_eur_per_tcs",
    "fuel_type",
    *ENERGY_INPUT_COLUMNS,
    "electricity_consumption_mwh_per_tcs",
    "electricity_price_eur_per_mwh",
    "emissions_tco2_per_tcs",
    "steel_price_eur_per_tcs",
    "carbon_price_eur_per_t",
)

RETROFIT_INPUT_COLUMNS = (
    "capex_change_eur_per_tcs",
    "fixed_opex_change_eur_per_tcs",
    "variable_opex_change_eur_per_tcs",
    "fuel_consumption_reduction_fraction",
    "electricity_consumption_reduction_fraction",
    "emissions_reduction_fraction",
)

ENERGY_OUTPUT_COLUMNS = {
    "bf_bof_bau": ("annual_pci_coking_coal_cost_eur",),
    "bf_bof_post_combustion_ccs": ("annual_pci_coking_coal_cost_eur",),
    "scrap_eaf": ("annual_charcoal_cost_eur",),
    "ng_dri_eaf_bau": ("annual_natural_gas_cost_eur",),
    "ng_dri_eaf_ccs": ("annual_natural_gas_cost_eur",),
    "h2_dri_eaf": (
        "annual_hydrogen_cost_eur",
        "annual_charcoal_cost_eur",
    ),
    "moe": (),
    "ael_eaf": ("annual_charcoal_cost_eur",),
}[TECHNOLOGY]

PROCESSED_OUTPUT_COLUMNS = (
    "initial_capex_eur",
    "annual_revenue_eur",
    "annual_fixed_opex_eur",
    "annual_variable_opex_eur",
    *ENERGY_OUTPUT_COLUMNS,
    "annual_fuel_cost_eur",
    "annual_electricity_cost_eur",
    "annual_emissions_cost_eur",
    "annual_total_cost_eur",
    "annual_net_cash_flow_eur",
    "npv_eur",
    "discounted_lifetime_output_tcs",
    "present_value_total_cost_eur",
    "lcos_eur_per_tcs",
    "levelized_net_margin_eur_per_tcs",
)

if value["technology_type"] == "retrofit":
    RAW_INPUT_COLUMNS = (
        *RAW_INPUT_COLUMNS,
        "transport_and_storage_share_of_capture_cost",
    )
    PROCESSED_OUTPUT_COLUMNS = (
        *PROCESSED_OUTPUT_COLUMNS[:6 + len(ENERGY_OUTPUT_COLUMNS)],
        "capture_cost_excluding_transport_and_storage_eur_per_tcs",
        "transport_and_storage_cost_eur_per_tcs",
        "annual_transport_and_storage_cost_eur",
        *PROCESSED_OUTPUT_COLUMNS[6 + len(ENERGY_OUTPUT_COLUMNS):],
    )

def build_table(columns):
    selected_columns = [column for column in columns if column in value]
    return pd.DataFrame(
        {
            "parameter": selected_columns,
            "value": [value[column] for column in selected_columns],
            "unit": [UNITS[column] for column in selected_columns],
        }
    )

summary = pd.DataFrame(
    {
        "metric": (
            "Technology",
            "Technology type",
            "NPV",
            "NPV",
            "LCOS",
            "Levelized net margin",
            "Annual net cash flow",
            "Discounted lifetime output",
        ),
        "value": (
            value["technology"],
            value["technology_type"],
            value["npv_eur"],
            value["npv_eur"] / 1_000_000,
            value["lcos_eur_per_tcs"],
            value["levelized_net_margin_eur_per_tcs"],
            value["annual_net_cash_flow_eur"],
            value["discounted_lifetime_output_tcs"],
        ),
        "unit": (
            "technology key",
            "category",
            "EUR",
            "MEUR",
            "EUR/tCS",
            "EUR/tCS",
            "EUR/year",
            "tCS",
        ),
    }
)
raw_inputs = build_table(RAW_INPUT_COLUMNS)
processed_outputs = build_table(PROCESSED_OUTPUT_COLUMNS)
if value["technology_type"] == "retrofit":
    retrofit_inputs = build_table(RETROFIT_INPUT_COLUMNS)

## Summary

In [3]:
summary

,metric,value,unit
0,Technology,bf_bof_post_combustion_ccs,technology key
1,Technology type,retrofit,category
2,NPV,"402,725,756.658",EUR
3,NPV,402.726,MEUR
4,LCOS,708.981,EUR/tCS
5,Levelized net margin,41.019,EUR/tCS
6,Annual net cash flow,"122,466,324.188",EUR/year
7,Discounted lifetime output,"9,818,147.407",tCS


## Expected inputs

In [4]:
raw_inputs

,parameter,value,unit
0,annual_output_tcs,"1,000,000.000",tCS/year
1,lifetime_years,20.000,years
2,technology_type,retrofit,category
3,capex_eur_per_tcs,799.667,EUR/tCS
4,fixed_opex_eur_per_tcs,50.100,EUR/tCS
5,variable_opex_eur_per_tcs,317.833,EUR/tCS
6,fuel_type,pci_coking_coal_mix,category
7,fuel_consumption_mwh_th_per_tcs,6.201,MWh_th/tCS
8,fuel_price_eur_per_mwh_th,23.000,EUR/MWh_th
9,electricity_consumption_mwh_per_tcs,0.443,MWh/tCS


## BAU-relative retrofit inputs

In [5]:
retrofit_inputs

,parameter,value,unit
0,capex_change_eur_per_tcs,204.000,EUR/tCS
1,fixed_opex_change_eur_per_tcs,7.100,EUR/tCS
2,variable_opex_change_eur_per_tcs,4.500,EUR/tCS
3,fuel_consumption_reduction_fraction,-0.110,fraction
4,electricity_consumption_reduction_fraction,-2.850,fraction
5,emissions_reduction_fraction,0.660,fraction


## Processed outputs

In [6]:
processed_outputs

,parameter,value,unit
0,initial_capex_eur,"799,666,666.667",EUR
1,annual_revenue_eur,"750,000,000.000",EUR/year
2,annual_fixed_opex_eur,"50,100,000.000",EUR/year
3,annual_variable_opex_eur,"317,833,333.333",EUR/year
4,annual_pci_coking_coal_cost_eur,"142,627,600.000",EUR/year
5,annual_fuel_cost_eur,"142,627,600.000",EUR/year
6,annual_electricity_cost_eur,"51,624,650.000",EUR/year
7,capture_cost_excluding_transport_and_storage_e...,84.728,EUR/tCS
8,transport_and_storage_cost_eur_per_tcs,15.844,EUR/tCS
9,annual_transport_and_storage_cost_eur,"15,844,092.479",EUR/year
